# Explore WMB-10X VIS Inhibitory Depth Filters

This notebook helps you:
- inspect WMB-10X metadata fields
- define inhibitory and depth filters in VIS
- load filtered cell x gene matrices (raw or log2)
- save reusable outputs for downstream comparison

In [5]:
from pathlib import Path
from typing import Iterable

import anndata as ad
import numpy as np
import pandas as pd

from abc_atlas_access.abc_atlas_cache.abc_project_cache import AbcProjectCache

ABC_ATLAS_DIR = Path('/root/capsule/data/abc_atlas')
OUT_DIR = Path('/root/capsule/scratch/reference_atlas_cellxgene/10x-hmb')
OUT_DIR.mkdir(parents=True, exist_ok=True)

HCR_PANEL_GENES = [
    'Calb1', 'Calb2', 'Cck', 'Chat', 'Crh', 'Gad2', 'Hpse', 'Lamp5',
    'Mme', 'Ndnf', 'Npy', 'Pdyn', 'Penk', 'Pthlh', 'Pvalb', 'Reln',
    'Slc17a7', 'Sst', 'Tac1', 'Tac2', 'Vip',
]

abc_cache = AbcProjectCache.from_cache_dir(ABC_ATLAS_DIR)
print('Cache ready:', ABC_ATLAS_DIR)

Cache ready: /root/capsule/data/abc_atlas


/opt/conda/lib/python3.12/site-packages/abc_atlas_access/abc_atlas_cache/cloud_cache.py:519: MissingLocalManifestWarning: This cache directory appears to contain data files, but it has no record of what those files are. Unless running as a LocalCache, files will be re-downloaded.
  warnings.warn(msg, MissingLocalManifestWarning)
/opt/conda/lib/python3.12/site-packages/abc_atlas_access/abc_atlas_cache/cloud_cache.py:1490: ReadOnlyLocalCacheWarning: LocalCache is a read only directory and cannot
                save the last used manifest.
                Current Manifest: releases/20260415/manifest.json
  warnings.warn(


In [6]:
cell = abc_cache.get_metadata_dataframe(
    directory='WMB-10X',
    file_name='cell_metadata_with_cluster_annotation',
    dtype={'cell_label': str},
)
cell = cell.set_index('cell_label')

print(f'Total WMB-10X cells: {len(cell):,}')
print(f'Metadata columns: {len(cell.columns)}')
display(pd.Series(cell.columns, name='column').to_frame().head(50))

candidate_depth_cols = [
    'parcellation_substructure',
    'parcellation_substructure_layer',
    'cortical_layer',
    'layer',
]
present_depth_cols = [c for c in candidate_depth_cols if c in cell.columns]
print('Depth-like columns present:', present_depth_cols)

for c in ['region_of_interest_acronym', 'class', 'subclass', 'supertype', 'cluster', 'neurotransmitter']:
    if c in cell.columns:
        print(f'\n{c} top values:')
        display(cell[c].value_counts(dropna=False).head(20).to_frame('n'))

/opt/conda/lib/python3.12/site-packages/abc_atlas_access/abc_atlas_cache/abc_project_cache.py:643: DtypeWarning: Columns (16) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path, **kwargs)


Total WMB-10X cells: 4,042,976
Metadata columns: 27


,column
0,cell_barcode
1,barcoded_cell_sample_label
2,library_label
3,feature_matrix_label
4,entity
5,brain_section_label
6,library_method
7,region_of_interest_acronym
8,donor_label
9,donor_genotype


Depth-like columns present: []

region_of_interest_acronym top values:


,n
region_of_interest_acronym,
MB,367029
VIS,321908
OLF,280744
HY,262175
TH,261009
MOp,247660
MY,192533
CB,182004
HIP,176122


...

,n
supertype,
1184 MOL NN_4,395554
1163 Astro-TE NN_3,146806
1160 Astro-NT NN_2,139205
0030 L2/3 IT CTX Glut_2,128897
1179 OPC NN_1,121667
1155 CB Granule Glut_2,116129
0028 L4/5 IT CTX Glut_6,90911
1193 Endo NN_1,88011
1194 Microglia NN_1,86232



cluster top values:


,n
cluster,
5285 MOL NN_4,264669
5225 Astro-TE NN_3,131046
5284 MOL NN_4,120642
5269 OPC NN_1,117304
5201 CB Granule Glut_2,115909
5214 Astro-NT NN_2,108101
0109 L2/3 IT CTX Glut_2,100893
5312 Microglia NN_1,86232
0100 L4/5 IT CTX Glut_6,72787



neurotransmitter top values:


,n
neurotransmitter,
Glut,2054137
NaN,1089152
GABA,834601
GABA-Glyc,36490
Dopa,9396
Glut-GABA,8989
Chol,7582
Sero,1469
Nora,626


In [7]:
# ---- Filter configuration ----
REGION = 'VIS'
EXPRESSION_TYPE = 'raw'  # 'raw' or 'log2'

# Keep only these inhibitory classes.
TARGET_CLASSES = ['06 CTX-CGE GABA', '07 CTX-MGE GABA']

# Drop supertypes whose name contains any of these tokens (case-insensitive).
EXCLUDE_SUPERTYPE_SUBSTRINGS = ['L6']

# Keep only supertypes with at least this many cells.
MIN_SUPERTYPE_CELLS = 10

# Picks first available depth-like metadata column from this list.
DEPTH_COL_CANDIDATES = [
    'parcellation_substructure',
    'parcellation_substructure_layer',
    'cortical_layer',
    'layer',
]

# Example VIS depths; adjust after viewing unique values below.
DEPTH_VALUES = ['VISp2/3', 'VISp4', 'VISp5', 'VISp6a', 'VISp6b']

# Genes to extract. Use HCR panel by default.
GENES = HCR_PANEL_GENES

# Optional output files
SAVE_PREFIX = f'10x_{REGION}_inh_cls_filtered_{EXPRESSION_TYPE}'
WRITE_PARQUET = True
WRITE_CSV = True

In [8]:
def choose_depth_column(df: pd.DataFrame, candidates: Iterable[str]) -> str | None:
    for c in candidates:
        if c in df.columns:
            return c
    return None


def filter_wmb10x_cells(
    cell_df: pd.DataFrame,
    region: str,
    target_classes: list[str] | None,
    exclude_supertype_substrings: list[str] | None,
    min_supertype_cells: int | None,
    depth_col_candidates: Iterable[str],
    depth_values: list[str] | None,
) -> tuple[pd.DataFrame, str | None]:
    region_cells = cell_df[cell_df['region_of_interest_acronym'] == region].copy()

    if target_classes is not None:
        if 'class' not in region_cells.columns:
            raise ValueError("'class' column not found in metadata.")
        region_cells = region_cells[region_cells['class'].isin(target_classes)].copy()

    if exclude_supertype_substrings:
        if 'supertype' not in region_cells.columns:
            raise ValueError("'supertype' column not found in metadata.")
        st = region_cells['supertype'].astype(str)
        exclude_mask = pd.Series(False, index=region_cells.index)
        for token in exclude_supertype_substrings:
            token = str(token).strip()
            if token:
                exclude_mask = exclude_mask | st.str.contains(token, case=False, na=False)
        region_cells = region_cells[~exclude_mask].copy()

    if min_supertype_cells is not None:
        if min_supertype_cells < 1:
            raise ValueError('min_supertype_cells must be >= 1')
        if 'supertype' not in region_cells.columns:
            raise ValueError("'supertype' column not found in metadata.")
        st_counts = region_cells['supertype'].value_counts()
        keep_st = st_counts[st_counts >= min_supertype_cells].index
        region_cells = region_cells[region_cells['supertype'].isin(keep_st)].copy()

    depth_col = choose_depth_column(region_cells, depth_col_candidates)
    if depth_col is not None and depth_values:
        region_cells = region_cells[region_cells[depth_col].isin(depth_values)].copy()

    return region_cells, depth_col


filtered_cells, depth_col_used = filter_wmb10x_cells(
    cell_df=cell,
    region=REGION,
    target_classes=TARGET_CLASSES,
    exclude_supertype_substrings=EXCLUDE_SUPERTYPE_SUBSTRINGS,
    min_supertype_cells=MIN_SUPERTYPE_CELLS,
    depth_col_candidates=DEPTH_COL_CANDIDATES,
    depth_values=DEPTH_VALUES,
)

print(f'Filtered cells: {len(filtered_cells):,}')
print('Depth column used:', depth_col_used)

if depth_col_used is not None:
    print('\nDepth distribution:')
    display(filtered_cells[depth_col_used].value_counts(dropna=False).to_frame('n').head(30))

for c in ['class', 'subclass', 'supertype', 'cluster', 'neurotransmitter']:
    if c in filtered_cells.columns:
        print(f'\n{c} distribution:')
        display(filtered_cells[c].value_counts(dropna=False).head(20).to_frame('n'))

Filtered cells: 47,030
Depth column used: None

class distribution:


,n
class,
06 CTX-CGE GABA,24840
07 CTX-MGE GABA,22190



subclass distribution:


,n
subclass,
046 Vip Gaba,12497
053 Sst Gaba,11926
049 Lamp5 Gaba,10074
052 Pvalb Gaba,9638
047 Sncg Gaba,2269
050 Lamp5 Lhx6 Gaba,355
051 Pvalb chandelier Gaba,271



supertype distribution:


,n
supertype,
0199 Lamp5 Gaba_1,6989
0207 Pvalb Gaba_3,5939
0217 Sst Gaba_4,3325
0177 Vip Gaba_5,2749
0178 Vip Gaba_6,2217
0200 Lamp5 Gaba_2,1754
0212 Pvalb Gaba_8,1504
0214 Sst Gaba_1,1485
0173 Vip Gaba_1,1479



cluster distribution:


,n
cluster,
0709 Lamp5 Gaba_1,4464
0741 Pvalb Gaba_3,2675
0742 Pvalb Gaba_3,1901
0645 Vip Gaba_6,1704
0754 Pvalb Gaba_8,1504
0740 Pvalb Gaba_3,1363
0708 Lamp5 Gaba_1,1348
0779 Sst Gaba_4,1263
0639 Vip Gaba_4,1151



neurotransmitter distribution:


,n
neurotransmitter,
GABA,46873
Glut-GABA,157


In [ ]:
def load_wmb10x_expression_for_cells(
    abc_cache,
    filtered_cell_meta: pd.DataFrame,
    genes: list[str] | None = None,
    expression_type: str = 'raw',
) -> pd.DataFrame:
    if expression_type not in {'raw', 'log2'}:
        raise ValueError("expression_type must be 'raw' or 'log2'")

    if len(filtered_cell_meta) == 0:
        raise ValueError('No cells after filtering.')

    if 'feature_matrix_label' not in filtered_cell_meta.columns or 'dataset_label' not in filtered_cell_meta.columns:
        raise ValueError("filtered_cell_meta must include 'feature_matrix_label' and 'dataset_label'")

    feature_labels = filtered_cell_meta['feature_matrix_label'].dropna().unique().tolist()
    parts = []
    found_genes = set()

    for feature_label in feature_labels:
        rows = filtered_cell_meta[filtered_cell_meta['feature_matrix_label'] == feature_label]
        dataset_dir = str(rows['dataset_label'].iloc[0])
        h5ad_path = abc_cache.get_file_path(
            directory=dataset_dir,
            file_name=f'{feature_label}/{expression_type}',
        )

        adata = ad.read_h5ad(h5ad_path, backed='r')
        common_cells = rows.index.intersection(adata.obs_names)
        if len(common_cells) == 0:
            adata.file.close()
            continue

        obs_mask = adata.obs_names.isin(common_cells)
        gene_meta = adata.var

        if 'gene_symbol' in gene_meta.columns:
            gene_symbols = gene_meta['gene_symbol'].astype(str)
        else:
            gene_symbols = pd.Series(gene_meta.index.astype(str), index=gene_meta.index)

        if genes:
            gene_mask = gene_symbols.isin(genes)
            selected_var_idx = gene_meta.index[gene_mask]
            selected_gene_symbols = gene_symbols[gene_mask].tolist()
            found_genes.update(selected_gene_symbols)

            if len(selected_var_idx) == 0:
                adata.file.close()
                continue

            part = adata[obs_mask, selected_var_idx].to_df()
            part.columns = selected_gene_symbols
        else:
            part = adata[obs_mask, :].to_df()
            part.columns = gene_symbols.tolist()

        # Merge duplicate gene symbols if present in var.
        part = part.groupby(part.columns, axis=1).sum()

        adata.file.close()
        parts.append(part)

    if len(parts) == 0:
        raise ValueError('No expression data loaded for filtered cells.')

    expr = pd.concat(parts, axis=0, join='outer').fillna(0)
    expr = expr.loc[~expr.index.duplicated(keep='first')]

    if genes:
        missing = sorted(set(genes) - found_genes)
        if missing:
            print(f'Missing genes filled with 0: {missing}')
        expr = expr.reindex(columns=genes, fill_value=0)

    # Align to filtered metadata index order where possible.
    keep_idx = filtered_cell_meta.index.intersection(expr.index)
    expr = expr.loc[keep_idx]

    return expr


expr = load_wmb10x_expression_for_cells(
    abc_cache=abc_cache,
    filtered_cell_meta=filtered_cells,
    genes=GENES,
    expression_type=EXPRESSION_TYPE,
)

print(f'Expression loaded: {expr.shape[0]:,} cells x {expr.shape[1]:,} genes')
display(expr.head(3))

In [ ]:
# Save filtered expression + metadata for reuse by batch pipelines / notebooks.
expr_out_parquet = OUT_DIR / f'{SAVE_PREFIX}_cell_x_gene.parquet'
expr_out_csv = OUT_DIR / f'{SAVE_PREFIX}_cell_x_gene.csv'
meta_out_csv = OUT_DIR / f'{SAVE_PREFIX}_cell_metadata.csv'

if WRITE_PARQUET:
    expr.to_parquet(expr_out_parquet)
    print('Saved:', expr_out_parquet)

if WRITE_CSV:
    expr.to_csv(expr_out_csv)
    print('Saved:', expr_out_csv)

meta_to_save = filtered_cells.loc[expr.index].copy()
meta_to_save.to_csv(meta_out_csv)
print('Saved:', meta_out_csv)

## Next Integration Step

Once filter settings are finalized, we can add matching CLI flags to `run_collect_reference_atlas_cellxgene.py` for:
- inhibitory-only toggle
- depth column and depth value filters
- saving a filtered 10x-hmb reference directly from batch runs